# Способ 1 в Colab: TPC-H на Polars GPU

Ноутбук проверяет GPU-путь проекта [gpu-analytics](https://github.com/neuraCollab/gpu-analytics) на бесплатной GPU Colab (обычно NVIDIA T4, 16 GB). Он прогоняет все 22 запроса TPC-H на DuckDB, Polars CPU и Polars GPU, сверяет ответы и показывает:
- **ускорение** Polars GPU относительно Polars CPU и лучшего CPU-движка;
- **список запросов, которые GPU-движок не выполнил**. Это откаты на CPU: из-за `raise_on_fail=True` они видны как ошибки, а не тихо пересчитываются на CPU.

**Важно про цифры.** У Colab слабый CPU: 2 виртуальных ядра. Поэтому сравнение «CPU против GPU» здесь нечестное в пользу GPU. Ноутбук нужен для отладки: работает ли GPU-путь и какие запросы поддерживаются. Числа для ВКР снимаются на ЕВК или арендованной машине с сильным CPU.

## Перед запуском
1. **Runtime → Change runtime type → T4 GPU** (или другая GPU) → Save.
2. Репозиторий приватный, поэтому Colab нужен токен GitHub:
   - на GitHub: **Settings → Developer settings → Fine-grained tokens** → токен с доступом **только к gpu-analytics**, право **Contents: Read-only**;
   - в Colab: значок ключа слева (**Secrets**) → добавить секрет `GH_TOKEN` со значением токена → включить **Notebook access**.

   Если репозиторий станет публичным, токен не нужен.
3. **Runtime → Run all.** Весь ноутбук на SF1 идёт около 5–10 минут.

## 1. Есть ли GPU

In [ ]:
# nvidia-smi — утилита драйвера NVIDIA: модель GPU, память, версия драйвера и CUDA.
# Если здесь ошибка «command not found» — GPU не включена (см. шаг 1 выше).
!nvidia-smi

## 2. Скачать код проекта

In [ ]:
import os  # Проверка, существует ли папка.
import subprocess  # Запуск git из Python.

REPO = "neuraCollab/gpu-analytics"  # Репозиторий на GitHub: «владелец/имя».

try:  # Пытаемся взять токен из секретов Colab...
    from google.colab import userdata  # Модуль доступа к секретам (есть только в Colab).
    token = userdata.get("GH_TOKEN")  # Значение секрета GH_TOKEN.
except Exception:  # ...секрета нет или к нему не дали доступ —
    token = None  # клонируем без токена (сработает, только если репозиторий публичный).

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"  # Адрес с токеном или без.
PROJECT = "/content/gpu-analytics"  # Полный путь: ячейку можно перезапускать из любой текущей папки.
# Вывод git не печатаем: в тексте ошибки мог бы оказаться адрес вместе с токеном.
if not os.path.exists(PROJECT):  # Первый запуск — клонируем.
    result = subprocess.run(["git", "clone", "--depth", "1", url, PROJECT], capture_output=True)  # --depth 1 — только последний коммит.
    # Убираем токен из настроек скачанного репозитория, чтобы он не лежал в .git/config.
    subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])
else:  # Папка уже есть (ячейку запускают повторно) — подтягиваем свежие изменения из GitHub.
    result = subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only", url, "main"], capture_output=True)  # --ff-only — только «перемотка вперёд», без слияний.
if result.returncode != 0:  # git завершился с ошибкой.
    raise RuntimeError("git не смог скачать код: проверьте секрет GH_TOKEN и доступ к нему (Notebook access)")

%cd {PROJECT}
!git log --oneline -1

## 3. Установить проект с GPU-движком
GPU-исполнитель Polars — пакет `cudf-polars` из NVIDIA RAPIDS. Он выпускается в двух вариантах: под CUDA 12 (`-cu12`) и под CUDA 13 (`-cu13`). В Colab RAPIDS уже предустановлен (cuDF, cuML), и ставить нужно **тот же вариант CUDA и ту же версию RAPIDS**. Иначе pip откатит общие CUDA-библиотеки и сломает предустановленные пакеты, в том числе torch. Ячейка ниже определяет это сама.

Это правило только для Colab. На GPU-сервере проект ставится обычной группой `pip install -e ".[gpu]"`, и там всегда CUDA 13.

**Если pip уже сообщал о конфликтах** (`cuda-toolkit … incompatible`) при запуске прошлой версии ноутбука, окружение испорчено: **Runtime → Disconnect and delete runtime**, затем снова Run all.

In [ ]:
import re  # Регулярные выражения: найти версию CUDA в выводе nvidia-smi.
from importlib import metadata  # Узнать версию установленного пакета, не импортируя его.


def version_of(package):
    """Версия установленного пакета или None, если его нет."""
    try:
        return metadata.version(package)  # Например, "26.6.0".
    except metadata.PackageNotFoundError:  # Пакет не установлен.
        return None


for cuda in ("13", "12"):  # Ищем предустановленный cuDF: сначала под CUDA 13, потом под CUDA 12.
    rapids = version_of(f"cudf-cu{cuda}")  # Версия RAPIDS, если такой cuDF есть.
    if rapids:  # Нашли — ставим cudf-polars того же варианта CUDA и той же версии «год.месяц» (26.6.*).
        GPU_PACKAGE = f"cudf-polars-cu{cuda}=={'.'.join(rapids.split('.')[:2])}.*"
        break  # Дальше не ищем.
else:  # Цикл прошёл без break — RAPIDS не предустановлен (не Colab): выбираем вариант по драйверу.
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout  # Вывод nvidia-smi.
    found = re.search(r"CUDA Version: (\d+)", smi)  # Максимальная версия CUDA, которую поддерживает драйвер.
    GPU_PACKAGE = f"cudf-polars-cu{found.group(1) if found else '12'}"  # cu13 или cu12 (по умолчанию cu12).
print("GPU-пакет:", GPU_PACKAGE)

# [dev] — pytest. Пакет cudf-polars сам подберёт совместимую версию Polars: GPU-движок поддерживает
# не самые свежие версии Polars, поэтому pip может поставить Polars старше, чем локально. Это нормально:
# Polars CPU и Polars GPU в этом ноутбуке работают на одной и той же версии, и сравнение честное.
# Установка не «редактируемая» (без -e): так пакет сразу виден в этом же ядре Python.
%pip install -q ".[dev]" "{GPU_PACKAGE}"

In [ ]:
# Окружение: CPU, RAM, GPU, версии библиотек. Тот же «паспорт» сохраняется рядом с каждым прогоном.
!python -m gpu_analytics.env_check

In [ ]:
# Самая простая проверка GPU-движка: сумма трёх чисел на GPU.
# Если здесь ошибка — дальше GPU-результатов не будет; текст ошибки подскажет причину (версия CUDA, драйвер, пакет).
import polars as pl  # Polars (уже новой версии, только что установленной).

check = pl.LazyFrame({"a": [1, 2, 3]}).select(pl.col("a").sum())  # Ленивый план: сумма колонки a.
print(check.collect(engine=pl.GPUEngine(raise_on_fail=True)))  # Выполнить на GPU; при неподдержке — ошибка, а не откат на CPU.
print("polars", pl.__version__)  # Версия Polars для журнала.

## 4. Сгенерировать TPC-H
`SF = 1` — около 1 GB, генерация занимает около минуты. Для SF10 поменяйте на `10`: генерация около 5 минут на CPU Colab, 3.2 GB Parquet. Таблица `lineitem` на SF10 может не поместиться в память T4 на тяжёлых запросах; такие ошибки — тоже результат (способ 4).

In [ ]:
SF = 1  # Масштаб данных: 1 ≈ 1 GB, 10 ≈ 10 GB.
DATA = f"data/tpch_sf{SF}"  # Папка с Parquet-файлами.
!python -m gpu_analytics.data.generate_tpch --sf {SF} --out {DATA}

## 5. Бенчмарк: DuckDB, Polars CPU, Polars GPU

In [ ]:
# Все 22 запроса на трёх движках: прогрев + 5 замеров, сверка каждого ответа с DuckDB.
# Результат — results/results_<время>.csv и паспорт окружения results/passport_<время>.json.
!python -m gpu_analytics.bench.run --data {DATA} --engines duckdb polars_cpu polars_gpu --out results

## 6. Ускорение и откаты на CPU

In [ ]:
import glob  # Поиск файлов по шаблону.
import os  # Время изменения файла (чтобы найти самый свежий).

import polars as pl  # Таблицы.

latest = max(glob.glob("results/results_*.csv"), key=os.path.getmtime)  # Самый свежий файл результатов.
df = pl.read_csv(latest)  # Читаем его.
print("Файл:", latest)

order = pl.col("query").str.slice(1).cast(pl.Int32)  # Номер запроса из имени «q12» → 12 (для сортировки q1, q2, …, q22).
times = (  # Таблица «запрос × движок → медиана времени, с».
    df.filter(pl.col("status") == "ok")  # Только успешно выполненные запуски.
    .pivot(on="engine", index="query", values="median_s")  # Движки — в колонки.
    .sort(order)  # Запросы по порядку номеров.
)
if "polars_gpu" in times.columns:  # Если хоть один запрос выполнился на GPU — считаем ускорение.
    times = times.with_columns(
        (pl.col("polars_cpu") / pl.col("polars_gpu")).round(2).alias("gpu_vs_polars_cpu"),  # Во сколько раз GPU быстрее Polars CPU.
        (pl.min_horizontal("duckdb", "polars_cpu") / pl.col("polars_gpu")).round(2).alias("gpu_vs_best_cpu"),  # …быстрее лучшего CPU-движка.
    )
with pl.Config(tbl_rows=30, float_precision=3):  # Показать все 22 строки, 3 знака после запятой.
    print(times)

failed = df.filter(pl.col("status") == "error").select("engine", "query", "error")  # Запросы, которые движок не выполнил.
print(f"\nНе выполнено: {failed.height}")  # Для polars_gpu это откаты: операции, которых нет в GPU-движке.
with pl.Config(fmt_str_lengths=200, tbl_rows=30):  # Показывать текст ошибки целиком (до 200 символов).
    print(failed)

wrong = df.filter(pl.col("correct") == False)  # Выполнились, но ответ не совпал с DuckDB.
print(f"\nНеверных ответов: {wrong.height}")  # Должно быть 0; иначе ускорение не засчитывается.
if wrong.height:
    print(wrong.select("engine", "query", "error"))

## 7. Тесты GPU

In [ ]:
# Тест сверяет Polars GPU с DuckDB на всех 22 запросах (SF 0.01). Неподдерживаемые на GPU запросы
# тест не валят, но всё, что выполнилось, обязано совпасть с DuckDB.
!python -m pytest -m gpu -q

## 8. (Необязательно) NYC Taxi на GPU
Скачивает поездки жёлтого такси за 2024 год (около 660 MB, 41 млн строк) и прогоняет 5 запросов. Чтобы включить ячейку, поставьте `RUN_NYC = True`.

In [ ]:
RUN_NYC = False  # Поменяйте на True, чтобы скачать данные и прогнать запросы.
if RUN_NYC:
    !python -m gpu_analytics.data.download_nyc_taxi --year 2024 --out data/nyc_taxi
    !python -m gpu_analytics.bench.run --suite nyc_taxi --data data/nyc_taxi --engines duckdb polars_cpu polars_gpu --out results

## 9. Сохранить результаты
Файлы Colab исчезают после отключения сессии, поэтому скачайте архив с результатами и паспортами окружения. Положите его содержимое в `results/colab/` локального репозитория и запишите итог в `docs/journal.md`.

In [ ]:
from google.colab import files  # Скачивание файлов из Colab на компьютер.

!zip -q -r results_colab.zip results
files.download("results_colab.zip")  # Браузер предложит сохранить архив.